<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/00_overview_and_video_gallery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 - Overview and video gallery

Welcome. This series teaches **S-JEPA**, a way for a model to learn what walking looks like without any labels, and then uses that learned sense of motion to tell apart three conditions from a short video of someone walking:

- **normal** gait
- **ms**: multiple sclerosis
- **pd**: Parkinson's disease

By the end you will have trained a small S-JEPA model on real walking clips, fine-tuned it across the three conditions, and compared it head to head against a classical Random Forest on the exact same videos.

This first notebook sets the scene. We look at the data, count it honestly, and actually watch a few clips so the later math stays grounded in real movement.

### How to run

You can run every notebook two ways:

1. **Locally** with `uv`. From the repo root: `cd experiments/multiple-sclerosis && uv sync`, then open the notebooks in Jupyter or VS Code.

2. **In Google Colab** by clicking the badge at the top. The setup cells install what is missing and clone the repo so `import sjepa` works.


In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

## The pipeline at a glance

Both the learned approach and the classical baseline start from the same pose front-end, then split into two branches, and finally meet again for a fair comparison.


In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(IMAGES_DIR / 'pipeline_flowchart.svg')))

## The dataset, counted honestly

The videos live in `video-data/` split into `normal`, `ms`, and `pd` folders. Some clips come from the same source video (a long YouTube clip cut into pieces). That matters a lot for a fair test, because two clips from one source are not independent. We track the **source id** now so later we can keep all clips of one source on the same side of a split.


In [ ]:
import pandas as pd
from sjepa.data import source_id_from_name

rows = []
for label in ['normal', 'ms', 'pd']:
    for vid in sorted((VIDEO_DIR / label).glob('*.mp4')):
        rows.append(dict(label=label, clip=vid.name, source_id=source_id_from_name(vid.name)))
manifest = pd.DataFrame(rows)
summary = manifest.groupby('label').agg(clips=('clip', 'count'),
                                        sources=('source_id', 'nunique'))
print(summary)
print('\ntotal clips:', len(manifest), '| total sources:', manifest.source_id.nunique())
manifest.to_csv(ARTIFACT_DIR / 'manifest_grouped.csv', index=False)

Notice that `pd` has many more clips than sources. That is the clip-splitting we will guard against. If we split clips at random, pieces of one walk could land in both training and testing and make the scores look better than they really are.

## Watch a few walks

Numbers are easier to trust once you have seen what they describe. The cell below embeds one clip per condition right in the notebook. Look for the differences the clinicians describe: normal gait is smooth and symmetric, ms gait can be unsteady with shorter steps, and pd gait often shows small shuffling steps and reduced arm swing.


In [ ]:
from sjepa.viz import show_video
from IPython.display import display

for label in ['normal', 'ms', 'pd']:
    clip = sorted((VIDEO_DIR / label).glob('*.mp4'))[0]
    print(f'{label}: {clip.name}')
    display(show_video(clip, width=360))

## Roadmap

| Notebook | What you build |
|---|---|
| 00 overview | this tour of the data and the plan |
| 01 pose extraction | turn videos into skeleton sequences with MediaPipe |
| 02 mask and tokens | the fixed anatomical mask and how skeletons become tokens |
| 03 pretrain on normal | build S-JEPA and train it on normal gait |
| 04 progressive fine-tune | add ms and pd, add VICReg to separate the classes |
| 05 representations | visualize the learned features with t-SNE and UMAP |
| 06 capstone | Random Forest vs S-JEPA on identical, leakage-safe splits |

On to notebook 01, where we turn these videos into skeletons.
